# Sequential Minimal Optimization

### For hard-margin SVM

We are going to optimize the following objective function of SVM:

$$
    \underset{\lambda = (\lambda_1, \lambda_2, ..., \lambda_N)^T}{max} \left[ \sum_{n=1}^N \lambda_n - \frac{1}{2}\sum_{i, j} \lambda_i \lambda_j a_i a_j K(x_i, x_j) \right]
$$
with respect to (w.r.t.) 
$$
    \sum_{n=1}^N \lambda_n a_n = 0, \quad a_i \ge 0
$$

The core idea of SMO is that instead of finding the whole vector $a$ at once, it selects two variables $(a_i, a_j)$ then optimize them analytically. Because of the constraints $\sum_{n=1}^N a_i y_i = 0$, whenever we change $a_i$, there must be another $a_j$ being updated.

The SMO method can be described step-by-step as follows:

- Step 1: Determine the prediction error
$$
    E_i = f(x_i) - y_i = \left( \sum_{k=1}^n a_k y_k K(x_i, x_k) + b \right) - y_i
$$

- Step 2: Pick two variables such that $a_i$ violates the KTT conditions and $a_j$ is chosen randomly (intuitively, we can pick $a_j$ so that $|E_i - E_j|$ is maximum).

The KKT conditions are given as below:

$$

\begin{align}
    \lambda_i > 0 & \Leftrightarrow & a_i y(x_n) = 1 \quad \text{(supported vectors)} \\
    \lambda_i = 0 & \Leftrightarrow & a_i y(x_n) \ge 1 \quad \text{(non-supported vectors)}
\end{align}

$$

- Step 3: Determine the lower bound $L$ and the higher bound $H$:

If $y_i \ne y_j$:

$$
    L = max(0, a_j - a_i), H = \infty
$$

If $y_i \ne y_j$:

$$
    L = max(0, a_i + a_j), H = \infty
$$

- Step 3: Compute $\eta$

$$
    \eta = K(x_i, x_i) + K(x_j, x_j) - 2K(x_i, x_j)
$$

- Step 4: Update $a_j$

$$
    a_j^{new} = a_j^{old} + \frac{y_j(E_i - E_j)}{\eta}
$$

$$
    a_j^{new} = max(L, a_j^{new})
$$

- Step 5: Update $a_i$

$$
    a_i = a_i + y_i y_j (a_j^{old} - a_j^{new})
$$

- Step 6: Update the bias

We have two candicates for the bias as follows

$$
    b_1 = b - E_i - y_i (a_i^{new} - a_i^{old}) K(x_i, x_i) - y_j (a_j^{new} - a_j^{old}) K(x_i, x_j)
$$

$$
    b_2 = b - E_j - y_i (a_i^{new} - a_i^{old}) K(x_i, x_j) - y_j (a_j^{new} - a_j^{old}) K(x_j, x_j)
$$

If $a_i^{new} > 0$, then 

$$
    b^{new} = b_1
$$

Else if $a_j^{new} > 0$, then 

$$
    b^{new} = b_2
$$

Otherwise, $b^{new} = \frac{1}{2}(b_1 + b_2)$.

Pseudo code for SMO on hard-margin SVM.

```python
initialize alpha = zeros(n)
b = 0

while not converged:
    for i in range(n):
        Ei = f(x_i) - y_i
        
        if violates_KKT(i):
            j = select_second_index(i)
            Ej = f(x_j) - y_j
            
            alpha_i_old = alpha[i]
            alpha_j_old = alpha[j]
            
            # Compute eta
            eta = K(i,i) + K(j,j) - 2*K(i,j)
            if eta <= 0:
                continue
            
            # Update alpha_j
            alpha[j] += y[j] * (Ei - Ej) / eta
            
            # Clip lower bound only
            alpha[j] = max(0, alpha[j])
            
            # Update alpha_i
            alpha[i] += y[i]*y[j]*(alpha_j_old - alpha[j])
            
            # Compute b1 and b2
            b1 = b - Ei \
                - y[i]*(alpha[i]-alpha_i_old)*K(i,i) \
                - y[j]*(alpha[j]-alpha_j_old)*K(i,j)
                
            b2 = b - Ej \
                - y[i]*(alpha[i]-alpha_i_old)*K(i,j) \
                - y[j]*(alpha[j]-alpha_j_old)*K(j,j)
            
            # Update b
            if alpha[i] > 0:
                b = b1
            elif alpha[j] > 0:
                b = b2
            else:
                b = 0.5*(b1 + b2)
```

## For soft-margin SVM

We are going to optimize the following objective function of SVM:

$$
    \underset{\lambda = (\lambda_1, \lambda_2, ..., \lambda_N)^T}{max} \left[ \sum_{n=1}^N \lambda_n - \frac{1}{2}\sum_{i, j} \lambda_i \lambda_j a_i a_j K(x_i, x_j) \right]
$$
with respect to (w.r.t.) 
$$
    \sum_{n=1}^N \lambda_n a_n = 0
$$

In the case of soft-margin SVM, we have the additional condition
$$
    0 \le a_i \le C
$$

The core idea of SMO is that instead of finding the whole vector $a$ at once, it selects two variables $(a_i, a_j)$ then optimize them analytically. Because of the constraints $\sum_{n=1}^N a_i y_i = 0$, whenever we change $a_i$, there must be another $a_j$ being updated.

The SMO method can be described step-by-step as follows:
- Step 1: Pick two variables such that $a_i$ violates the KTT conditions and $a_j$ is chosen randomly.

The KKT conditions are given as below:

$$

\begin{align}
    a_i = 0 & \Leftrightarrow & y_i f(x_i) \ge 1 \quad \text{(correctly assigned vectors)} \\
    0 \le \alpha_i \le C & \Leftrightarrow & y_i f(x_i) = 1 \quad \text{(support vectors)} \\
    \alpha_i = C & \Leftrightarrow & y_i f(x_i) \le 1 \quad \text{(wrongly assigned vectors)}
\end{align}

$$

- Step 2: Determine the prediction error
$$
    E_i = f(x_i) - y_i = \left( \sum_{k=1}^n a_k y_k K(x_i, x_k) + b \right) - y_i
$$

- Step 3: Specifying the low (L) and high (H) boundaries

If $y_i \ne y_j$

$$
\begin{align}
    L = max(0, \alpha_j - \alpha_i) \\
    H = min(C, C + \alpha_j - \alpha_i)
\end{align}
$$

If $y_i = y_j$

$$
\begin{align}
    L = max(0, \alpha_j + \alpha_i - C) \\
    H = min(C, \alpha_j + \alpha_i)
\end{align}
$$

-Step 4: Compute $\eta$

$$
    \eta = K(x_i, x_i) + K(x_j, x_j) - 2K(x_i, x_j)
$$

- Step 5: Update $a_j$

$$
    a_j^{new} = a_j^{old} + \frac{y_j(E_i - E_j)}{\eta}
$$

Then we clip the value of $a_j$ so that $a_j \in [L, H]$

- Step 6: Update $a_i$

$$
    a_i = a_i + y_i y_j (a_j^{old} - a_j^{new})
$$

- Step 7: Update the bias

We have two candicates for the bias as follows

$$
    b_1 = b - E_i - y_i (a_i^{new} - a_i^{old}) K(x_i, x_i) - y_j (a_j^{new} - a_j^{old}) K(x_i, x_j)
$$

$$
    b_2 = b - E_j - y_i (a_i^{new} - a_i^{old}) K(x_i, x_j) - y_j (a_j^{new} - a_j^{old}) K(x_j, x_j)
$$

If $a_i$ is not at the boundary, then 

$$
    b^{new} = b_1
$$

Else if $a_j$ is not at the boundary, then 

$$
    b^{new} = b_2
$$

Otherwise, $b^{new} = \frac{1}{2}(b_1 + b_2)$.